# C4 V2 — Global + Six ROI — fixed train/validation split
Bật **T4 GPU**, sau đó chọn **Runtime → Run all**. Không sửa cell. Mỗi bước được tách riêng để dễ theo dõi.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv
import os
import shutil
import subprocess
import sys
import zipfile

DRIVE_DATA = Path('/content/drive/MyDrive/data')
WORKSPACE = Path('/content/c4_multi_roi_v2_workspace')
GLOBAL_ZIP = DRIVE_DATA / 'data_dev_v1.zip'
ROI_ZIP = DRIVE_DATA / 'C4_MULTI_ROI_V2_DATA.zip'
CODE_ZIP = DRIVE_DATA / 'C4_MULTI_ROI_V2_COLAB_CODE.zip'
CONFIG = Path('c4_multi_roi/configs/fixed_split_colab.toml')
print('Drive:', DRIVE_DATA)
print('Workspace:', WORKSPACE)

In [ ]:
required = (GLOBAL_ZIP, ROI_ZIP, CODE_ZIP)
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Thiếu file trên Drive:\n' + '\n'.join(missing)
for path in required:
    print(f'OK {path.name}: {path.stat().st_size / (1024**3):.3f} GiB')

In [ ]:
if WORKSPACE.exists():
    assert WORKSPACE == Path('/content/c4_multi_roi_v2_workspace')
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

def extract_zip(path: Path, destination: Path) -> None:
    print(f'Đang giải nén {path.name} ...', flush=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(path) as archive:
        members = archive.infolist()
        for index, member in enumerate(members, start=1):
            target = (destination / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise RuntimeError(f'ZIP chứa đường dẫn không an toàn: {member.filename}')
            archive.extract(member, destination)
            if index % 5000 == 0 or index == len(members):
                print(f'  {index}/{len(members)} files', flush=True)

extract_zip(CODE_ZIP, WORKSPACE)
print('Code: DONE')

In [ ]:
extract_zip(ROI_ZIP, WORKSPACE)
print('ROI V2: DONE')

In [ ]:
extract_zip(GLOBAL_ZIP, WORKSPACE)
print('Global images: DONE')

In [ ]:
manifest = WORKSPACE / 'c4_multi_roi/cache/C4_MULTI_ROI_V2/manifest.csv'
with manifest.open('r', encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
for index, row in enumerate(rows, start=1):
    source = WORKSPACE / row['global_source_path']
    destination = WORKSPACE / row['global_path']
    assert source.is_file(), f'Thiếu ảnh whole nguồn: {source}'
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() or destination.is_symlink():
        destination.unlink()
    os.symlink(source, destination)
    if index % 2000 == 0 or index == len(rows):
        print(f'Linked 00_whole.png: {index}/{len(rows)}', flush=True)
assert len(rows) == 14024
print('Cấu trúc train/validation theo ID: DONE')

In [ ]:
os.chdir(WORKSPACE)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     'c4_multi_roi/requirements_colab.txt'],
    check=True,
)
print('Dependencies: DONE')

In [ ]:
result = subprocess.run(
    [sys.executable, '-u', '-m', 'c4_multi_roi.preflight',
     '--config', str(CONFIG), '--quick'],
    cwd=WORKSPACE,
    check=False,
)
assert result.returncode == 0, 'Preflight FAIL — dừng trước khi train'
print('Preflight: PASS')

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'CUDA chưa bật. Chọn Runtime > Change runtime type > T4 GPU rồi Run all lại.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

In [ ]:
subprocess.run(
    [sys.executable, '-u', '-m', 'c4_multi_roi.colab_runner',
     '--config', str(CONFIG)],
    cwd=WORKSPACE,
    check=True,
)

In [ ]:
import json
run_dir = DRIVE_DATA / 'c4_multi_roi_v2_runs' / 'C4_MULTI_ROI_V2_FIXED_SPLIT'
state = run_dir / 'run_state.json'
print('Saved at:', run_dir)
print(json.loads(state.read_text(encoding='utf-8')) if state.is_file()
      else 'run_state.json chưa được ghi')